<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/25_conversational_rag_memory/conversational_rag_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers scikit-learn pandas numpy

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
documents = [
    "Elon Musk founded SpaceX in 2002.",
    "SpaceX is a private aerospace company focused on space exploration.",
    "Elon Musk is also the CEO of Tesla.",
    "Tesla is an electric vehicle company.",
    "Tesla is headquartered in Austin, Texas."
]

df = pd.DataFrame({"text": documents})
df

,text
0,Elon Musk founded SpaceX in 2002.
1,SpaceX is a private aerospace company focused ...
2,Elon Musk is also the CEO of Tesla.
3,Tesla is an electric vehicle company.
4,"Tesla is headquartered in Austin, Texas."


In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embedding_model.encode(df['text'].tolist())

In [ ]:
def retrieve(query, top_k=2):
    query_embedding = embedding_model.encode([query])
    scores = cosine_similarity(query_embedding, doc_embeddings)[0]
    top_indices = np.argsort(scores)[-top_k:][::-1]
    return df.iloc[top_indices]

In [ ]:
chat_history = []

def build_context_from_history():
    context = ""
    for q, a in chat_history:
        context += f"User: {q}\nBot: {a}\n"
    return context

In [ ]:
def clean_context(context):
    sentences = context.split(".")
    unique = []
    for s in sentences:
        s = s.strip()
        if s and s not in unique:
            unique.append(s)
    return ". ".join(unique)

In [ ]:
def extract_answer(context, query):
    context_lower = context.lower()

    if "who founded spacex" in query.lower():
        return "Elon Musk"

    if "what company does he run" in query.lower():
        if "tesla" in context_lower and "spacex" in context_lower:
            return "Tesla and SpaceX"

    return "Answer not found"

In [ ]:
def conversational_rag(query):
    print("\n🔹 User Query:", query)

    history_context = build_context_from_history()

    combined_query = history_context + " " + query

    docs = retrieve(combined_query)
    retrieved_context = " ".join(docs['text'].tolist())

    final_context = clean_context(history_context + " " + retrieved_context)

    print("\n🧠 Context Used:\n", final_context)

    answer = extract_answer(final_context, query)

    print("\n✅ Answer:", answer)

    chat_history.append((query, answer))

    return answer

In [12]:
conversational_rag("Who founded SpaceX?")
conversational_rag("What company does he run?")


🔹 User Query: Who founded SpaceX?

🧠 Context Used:
 User: Who founded SpaceX?
Bot: Elon Musk
User: What company does he run?
Bot: Tesla and SpaceX
 Elon Musk founded SpaceX in 2002. Elon Musk is also the CEO of Tesla

✅ Answer: Elon Musk

🔹 User Query: What company does he run?

🧠 Context Used:
 User: Who founded SpaceX?
Bot: Elon Musk
User: What company does he run?
Bot: Tesla and SpaceX
User: Who founded SpaceX?
Bot: Elon Musk
 Elon Musk is also the CEO of Tesla. Elon Musk founded SpaceX in 2002

✅ Answer: Tesla and SpaceX


'Tesla and SpaceX'